# Visualize Rates
### Import libraries

In [ ]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm
%matplotlib qt

In [ ]:
from utils import add_package_path

Paths

In [ ]:
# Current Working Directory
cwd = os.getcwd()
# WORKSPACE
ws = os.environ['WORKSPACE_PATH']

In [ ]:
prefix = '/home/zanardi/Workspace/ML/PyCOMET'
paths = {
    # PyCOMET source dir
    'pycomet': prefix + '/pycomet/',
    # SurQCT source dir
    'surqct': prefix + '/pycomet/examples/SurQCT/'
}
add_package_path(paths)

In [ ]:
# Output directory
out = cwd + '/Output/visualizeRates/Inel/final/c03.r01/'
if not os.path.exists(out):
    os.makedirs(out)

### Inputs

In [ ]:
SystemName   = 'O3_UMN'
RatesId      = 'Inel'
TTranVec     = [5000]
MinRateValue = 1.e-16
iIdxVec      = np.array([1, 10, 32, 34, 35, 81, 82, 98, 1000, 2000, 3000, 4581, 6115])-1

SuffixesHDF5 = {
    'QCT':    '/Orig/',
    'SurQCT': '/final/ref/'
}

### Initialize SurQCT parameters

Import additional modules

In [ ]:
from surqct.src.chemical_systems import systems as syst
from surqct.src.rates_type.rates_type import RatesType
from surqct import config as cfg

Initializing QCT & SurQCT chemical systems

In [ ]:
Systems = {}
for Identifier, SuffixHDF5 in SuffixesHDF5.items():
    # Initializing rates
    Systems[Identifier] = syst.get_system(
        NameLong=SystemName,
        KinSource=Identifier,
        ProcNames=tf.nest.flatten(RatesId),
        PathToHDF5Fld=cfg.paths.HDF5Dtb + '/' + Identifier + '/' + SuffixHDF5
    )
    # Reading rates
    Systems[Identifier].Processes.read_rates(TTranVec=TTranVec, MicroRev=True)

### Plotting rates

In [ ]:
rates_type = RatesType(Systems['QCT'])
for iIdx in tqdm(iIdxVec):
    for i, TTran in enumerate(TTranVec):
        TTranStr = str(int(TTran))
        TrueData = Systems['QCT'].Processes.Rates[RatesId][TTranStr][iIdx]
        PredData = Systems['SurQCT'].Processes.Rates[RatesId][TTranStr][iIdx]
        jIdxTrue = np.where(TrueData > MinRateValue)[0]
        jIdxPred = np.where(PredData > MinRateValue)[0]
        rates_type.plot_prediction3D(
            RatesType=RatesId,
            Path=out,
            iIdx=iIdx,
            TrueData=(jIdxTrue, TrueData[jIdxTrue]),
            PredData=(jIdxPred, PredData[jIdxPred])
        )